In [3]:
from sklearn.ensemble import RandomForestClassifier

In [4]:
import joblib
X_train = joblib.load("../data/X_train_raw.joblib")
X_test = joblib.load("../data/X_test_raw.joblib")

y_train = joblib.load("../data/y_train.joblib")
y_test = joblib.load("../data/y_test.joblib")

In [5]:
y_train = y_train.map({
    "Rejected": 0,
    "Approved": 1
})

y_test = y_test.map({
    "Rejected": 0,
    "Approved": 1
})

In [6]:
print(y_train.isnull().sum())
print(y_test.isnull().sum())

0
0


In [7]:
print(y_train.value_counts())

 loan_status
1    2125
0    1290
Name: count, dtype: int64


In [8]:
numerical_features = [
    "no_of_dependents",
    " income_annum",
    " loan_amount",
    " loan_term",
    " cibil_score",
    " residential_assets_value",
    " commercial_assets_value",
    " luxury_assets_value",
    " bank_asset_value",
    " total_assets",
    " loan_to_income_ratio",
    " asset_to_loan_ratio",
    " asset_to_income_ratio"
]

categorical_features = [
    " education",
    " self_employed"
]

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

In [11]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

In [12]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
from sklearn.model_selection import cross_validate

rf_cv_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    return_train_score=True,
    n_jobs=-1
)

In [14]:
print("Mean CV Accuracy :", rf_cv_results["test_accuracy"].mean())
print("Mean CV Precision:", rf_cv_results["test_precision"].mean())
print("Mean CV Recall   :", rf_cv_results["test_recall"].mean())
print("Mean CV F1       :", rf_cv_results["test_f1"].mean())
print("Mean CV ROC-AUC  :", rf_cv_results["test_roc_auc"].mean())

Mean CV Accuracy : 0.9979502196193264
Mean CV Precision: 0.997191873985345
Mean CV Recall   : 0.999529411764706
Mean CV F1       : 0.9983576318048005
Mean CV ROC-AUC  : 0.9998832649338805


In [15]:
print("Mean Train Accuracy :", rf_cv_results["train_accuracy"].mean())
print("Mean Train Precision:", rf_cv_results["train_precision"].mean())
print("Mean Train Recall   :", rf_cv_results["train_recall"].mean())
print("Mean Train F1       :", rf_cv_results["train_f1"].mean())
print("Mean Train ROC-AUC  :", rf_cv_results["train_roc_auc"].mean())

Mean Train Accuracy : 1.0
Mean Train Precision: 1.0
Mean Train Recall   : 1.0
Mean Train F1       : 1.0
Mean Train ROC-AUC  : 1.0


# Tune n_estimators

In [16]:
from sklearn.model_selection import GridSearchCV

rf_param_grid = {
    "model__n_estimators": [100, 200, 300, 500]
}

rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

rf_grid_search.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__n_estimators': [100, 200, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('num', ...), ('cat', ...)]"


In [17]:
print("Best n_estimators:", rf_grid_search.best_params_)
print("Best CV ROC-AUC:", rf_grid_search.best_score_)

Best n_estimators: {'model__n_estimators': 200}
Best CV ROC-AUC: 0.9998932968536252


# Tune max_depth

from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__max_depth": [None, 3, 5, 7, 10, 15, 20]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best max_depth:", rf_grid.best_params_)
print("Best CV ROC-AUC:", rf_grid.best_score_)

# min_samples_split

In [18]:
param_grid = {
    "model__n_estimators": [200],
    "model__max_depth": [15],
    "model__min_samples_split": [2, 5, 10, 20, 30]
}

rf_grid_split = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

rf_grid_split.fit(X_train, y_train)

print("Best parameters:", rf_grid_split.best_params_)
print("Best CV ROC-AUC:", rf_grid_split.best_score_)

Best parameters: {'model__max_depth': 15, 'model__min_samples_split': 20, 'model__n_estimators': 200}
Best CV ROC-AUC: 0.9999452804377565


# min_samples_leaf

In [19]:
param_grid = {
    "model__n_estimators": [200],
    "model__max_depth": [15],
    "model__min_samples_split": [20],
    "model__min_samples_leaf": [1, 2, 4, 5, 10]
}

rf_grid_leaf = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

rf_grid_leaf.fit(X_train, y_train)

print("Best parameters:", rf_grid_leaf.best_params_)
print("Best CV ROC-AUC:", rf_grid_leaf.best_score_)

Best parameters: {'model__max_depth': 15, 'model__min_samples_leaf': 1, 'model__min_samples_split': 20, 'model__n_estimators': 200}
Best CV ROC-AUC: 0.9999452804377565


# max_features

In [20]:
param_grid = {
    "model__n_estimators": [200],
    "model__max_depth": [15],
    "model__min_samples_split": [20],
    "model__min_samples_leaf": [1],
    "model__max_features": ["sqrt", "log2", None]
}


rf_grid_features = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

rf_grid_features.fit(X_train, y_train)

print("Best parameters:", rf_grid_features.best_params_)
print("Best CV ROC-AUC:", rf_grid_features.best_score_)

Best parameters: {'model__max_depth': 15, 'model__max_features': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 20, 'model__n_estimators': 200}
Best CV ROC-AUC: 0.9999717282261742


# Create the final model

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_final = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=20,
        min_samples_leaf=1,
        max_features=None,
        random_state=42,
        n_jobs=-1
    ))
])

In [22]:
rf_final.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [23]:
y_pred_rf = rf_final.predict(X_test)
y_prob_rf = rf_final.predict_proba(X_test)[:, 1]

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Random Forest Test Accuracy :",
      accuracy_score(y_test, y_pred_rf))

print("Random Forest Test Precision:",
      precision_score(y_test, y_pred_rf))

print("Random Forest Test Recall   :",
      recall_score(y_test, y_pred_rf))

print("Random Forest Test F1       :",
      f1_score(y_test, y_pred_rf))

print("Random Forest Test ROC-AUC  :",
      roc_auc_score(y_test, y_prob_rf))

Random Forest Test Accuracy : 0.9988290398126464
Random Forest Test Precision: 0.9981203007518797
Random Forest Test Recall   : 1.0
Random Forest Test F1       : 0.9990592662276576
Random Forest Test ROC-AUC  : 0.9999941695381691


In [25]:
from sklearn.metrics import confusion_matrix

cm_rf = confusion_matrix(y_test, y_pred_rf)

print(cm_rf)

[[322   1]
 [  0 531]]


In [26]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       323
           1       1.00      1.00      1.00       531

    accuracy                           1.00       854
   macro avg       1.00      1.00      1.00       854
weighted avg       1.00      1.00      1.00       854



In [28]:
print(rf_grid_features.best_estimator_)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['no_of_dependents',
                                                   ' income_annum',
                                                   ' loan_amount', ' loan_term',
                                                   ' cibil_score',
                                                   ' residential_assets_value',
                                                   ' commercial_assets_value',
                                                   ' luxury_assets_value',
                                                   ' bank_asset_value',
                                                   ' total_assets',
                                                   ' loan_to_income_ratio',
                                                   ' asset_to_loan_ratio',
                                                   ' asset_to_income_ratio']),
 

In [ ]:
import joblib

joblib.dump(
    rf_grid.best_estimator_,
    "../models/random_forest_final_pipeline.pkl"
)